# Inventory Root Cause Analysis Engine

## 1. Project Overview

This notebook is a **Root Cause Analysis (RCA) Engine** for inventory risk.

It picks up **after** inventory risk has already been assessed. The Risk
Score, Risk Level, and all engineered inventory features (coverage,
sufficiency, safety stock, allocation, demand pressure, lead time
exposure, stock gap, and shortage exposure) are treated as **finished,
trusted inputs**. This notebook does not recalculate, adjust, or second-
guess any of those values.

Its only job is to answer one business question for every Medium,
High, and Critical risk item:

> **"Why is this inventory record High Risk — and what should we do about it?"**

The output is written in the language of Supply Chain Managers,
Procurement, Warehouse Operations, ERP users, and Executives — not in the
language of statistics or machine learning. It is designed to sit
alongside tools like SAP IBP, Oracle SCM Cloud, Dynamics 365 Supply
Chain, or Kinaxis as an explainability layer on top of an existing risk
model.

## 2. Notebook Objectives

- Load the already-scored inventory dataset and validate it, without
  touching any risk calculation.
- Build a **Knowledge Base** that describes what each engineered feature
  means in business terms.
- Build a **Root Cause Analysis Engine** that determines, per record, how
  much each feature is contributing to the current risk (`Critical
  Contributor` → `No Contribution`).
- Build a **Business Language Engine** that turns each contributing
  feature into a technical explanation, a plain-language human
  explanation, a business impact statement, and recommended actions —
  always referencing the record's actual values.
- Combine everything into a single **Root Cause Report** per inventory
  record, plus an **Executive Summary**.
- Validate every record defensively (missing values, invalid ranges) and
  log warnings instead of crashing.
- Run the engine as a **batch process** over every Medium / High /
  Critical risk record and export the results to
  `inventory_root_cause_analysis.csv`.

**What this notebook explicitly does *not* do:** recompute Risk Score,
Risk Level, AHP weights, or any of the business rules that produced
them. Those are inputs, not outputs, of this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
cd /content/drive/MyDrive/SiaCore

/content/drive/MyDrive/SiaCore


## 3. Imports

In [ ]:
from __future__ import annotations

import logging
import warnings
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Callable, Dict, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

## 4. Configuration

All paths, column names, and thresholds used by the engine are defined
in one place so the notebook is easy to adapt to a new export from the
upstream Inventory Risk Assessment notebook.

In [ ]:
# =============================================================================
# Project Configuration
# =============================================================================

PROJECT_NAME = "Inventory Root Cause Analysis Engine"
VERSION = "1.0.0"

# Candidate input files, checked in order. Update this list (or point
# INPUT_FILE directly at a path) to match wherever the upstream Inventory
# Risk Assessment notebook wrote its results.
INPUT_FILE_CANDIDATES = [
    Path("inventory_rows__2_.csv"),
    Path("inventory_rows (2).csv"),
    Path("/mnt/user-data/uploads/inventory_rows__2_.csv"),
]

OUTPUT_DIR = Path("/content/drive/MyDrive/SiaCore/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
EXPORT_FILE = OUTPUT_DIR / "inventory_root_cause_analysis_V2.csv"

# Risk levels that are in scope for Root Cause Analysis. Records outside
# this list are considered healthy and are skipped by the batch engine.
HIGH_RISK_LEVELS = ["High", "Critical","Medium"]
KNOWN_RISK_LEVELS = ["Very Low", "Low", "Medium", "High", "Critical"]

# Identifier and risk columns expected to already exist in the dataset.
ID_COLUMNS = ["inventory_id", "mpn"]
RISK_COLUMNS = ["risk_score", "risk_level"]

# The eight engineered features this engine explains. These must already
# exist in the input file — they are never recalculated here.
ENGINEERED_FEATURES = [
    "inventory_coverage_days",
    "stock_sufficiency_ratio",
    "safety_stock_ratio",
    "allocation_ratio",
    "demand_pressure",
    "lead_time_exposure",
    "stock_gap",
    "shortage_exposure",
]

REQUIRED_COLUMNS = ID_COLUMNS + RISK_COLUMNS + ENGINEERED_FEATURES

# Maximum number of root causes surfaced per record, to keep reports
# focused on the drivers that matter most rather than an exhaustive list.
MAX_ROOT_CAUSES_PER_RECORD = 4
# --- NEW CONFIGURATION ADDITIONS ---
INVENTORY_DATE_COLUMN = "inventory_date"   # Change to your actual column name if different
INCLUDE_FINANCIAL_IMPACT = True            # Toggle to True to add financial summary

## 5. Logging

In [ ]:
# =============================================================================
# Logging Configuration
# =============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("InventoryRCA")
logger.info(f"{PROJECT_NAME} v{VERSION} initialized")

## 6. Data Loading & Validation

We load the already-scored inventory dataset produced by the upstream
Inventory Risk Assessment notebook and verify that every column this
engine depends on is present. If required columns are missing, the
notebook stops here with a clear error rather than generating incorrect
explanations later.

In [ ]:
# =============================================================================
# Load Inventory Risk Assessment Results
# =============================================================================

def resolve_input_file(candidates: List[Path]) -> Path:
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"None of the candidate input files were found: {candidates}. "
        f"Update INPUT_FILE_CANDIDATES in the Configuration section."
    )


INPUT_FILE = resolve_input_file(INPUT_FILE_CANDIDATES)
inventory_df = pd.read_csv(INPUT_FILE)

logger.info(f"Loaded '{INPUT_FILE}' — {len(inventory_df):,} records, "
            f"{inventory_df.shape[1]} columns")

In [ ]:
# =============================================================================
# Validate Required Columns
# =============================================================================

missing_columns = [c for c in REQUIRED_COLUMNS if c not in inventory_df.columns]

if missing_columns:
    raise ValueError(
        f"Input file is missing required columns: {missing_columns}. "
        f"This engine reuses risk output as-is and cannot proceed without them."
    )

unknown_risk_levels = sorted(
    set(inventory_df["risk_level"].dropna().unique()) - set(KNOWN_RISK_LEVELS)
)
if unknown_risk_levels:
    logger.warning(f"Unexpected risk_level values found: {unknown_risk_levels}")

logger.info("Data validation passed — all required columns present")
inventory_df[["inventory_id", "mpn", "risk_score", "risk_level"] + ENGINEERED_FEATURES].head(3)

,inventory_id,mpn,risk_score,risk_level,inventory_coverage_days,stock_sufficiency_ratio,safety_stock_ratio,allocation_ratio,demand_pressure,lead_time_exposure,stock_gap,shortage_exposure
0,INV00001,PIC16F15245TI/SS,48.129316,Low,94.170020,0.442941,0.637977,0.266226,0.479539,342870.45,13834,27064.083350
1,INV00002,MCP3008TI/SL,49.200371,Low,108.128342,0.236990,0.321565,0.180713,0.341276,41403.60,3255,6135.806502
2,INV00003,LM384N/NOPB,55.906943,Medium,23.968231,0.057454,0.080653,0.542005,2.919657,592106.40,27725,97771.569300


## 7. Reference Statistics *(internal use only)*

Before an individual record can be explained, the engine needs a sense of
what a "typical" value looks like across the portfolio, so it can tell a
below-average coverage figure apart from a genuinely alarming one.

The table below is a **technical, analyst-facing reference table**. It is
built once from the full dataset and used internally by the Root Cause
Analysis Engine to grade each feature's contribution. **None of these
statistical values (median, quartiles, percentiles) are ever surfaced in
the generated business explanations** — those are written entirely in
plain business language, as required for this notebook to be usable by
non-technical stakeholders.

In [ ]:
# =============================================================================
# Build Reference Distribution (internal engine input — not user-facing)
# =============================================================================

@dataclass
class FeatureDistribution:
    p10: float
    q1: float
    median: float
    q3: float
    p90: float


def build_reference_distribution(
    data: pd.DataFrame, features: List[str]
) -> Dict[str, FeatureDistribution]:
    """Compute the internal distribution markers used to grade how far a
    record's feature value sits from the rest of the portfolio. These
    values power the contribution engine only and are never printed as
    part of a business explanation."""
    distribution = {}
    for feature in features:
        series = data[feature].dropna()
        distribution[feature] = FeatureDistribution(
            p10=float(series.quantile(0.10)),
            q1=float(series.quantile(0.25)),
            median=float(series.quantile(0.50)),
            q3=float(series.quantile(0.75)),
            p90=float(series.quantile(0.90)),
        )
    return distribution


REFERENCE_DISTRIBUTION = build_reference_distribution(inventory_df, ENGINEERED_FEATURES)

# Analyst-facing view only — internal reference table, not part of any
# generated inventory record explanation.
reference_table = pd.DataFrame({k: vars(v) for k, v in REFERENCE_DISTRIBUTION.items()}).T
reference_table.round(2)

,p10,q1,median,q3,p90
inventory_coverage_days,16.63,41.41,77.21,204.47,878.59
stock_sufficiency_ratio,0.04,0.14,0.37,1.27,2.55
safety_stock_ratio,0.05,0.19,0.53,1.79,3.66
allocation_ratio,0.10,0.21,0.37,0.58,0.69
demand_pressure,0.03,0.12,0.46,0.89,1.71
lead_time_exposure,22358.02,43703.65,102896.00,217461.46,399575.70
stock_gap,-12737.30,-1474.25,3072.00,12013.25,24058.00
shortage_exposure,1622.23,4280.23,11281.86,30909.13,72008.98


## 8. Root Cause Knowledge Base

The Knowledge Base defines, for every engineered feature, what it means
in business terms, which direction is risky, and how much relative
weight it carries when the engine ranks root causes. This weight is used
**only to prioritize which explanations to surface first** — it is not a
risk-scoring formula, and it does not change the Risk Score or Risk
Level that were already calculated upstream.

In [ ]:
# =============================================================================
# Direction & Contribution Enums
# =============================================================================

class Direction(str, Enum):
    LOWER_IS_RISKIER = "lower_is_riskier"
    HIGHER_IS_RISKIER = "higher_is_riskier"


class ContributionLevel(str, Enum):
    CRITICAL = "Critical Contributor"
    HIGH = "High Contributor"
    MODERATE = "Moderate Contributor"
    MINOR = "Minor Contributor"
    NONE = "No Contribution"


# Used only to rank contributors within a record — not a probability or score.
_SEVERITY_RANK = {
    ContributionLevel.CRITICAL: 4,
    ContributionLevel.HIGH: 3,
    ContributionLevel.MODERATE: 2,
    ContributionLevel.MINOR: 1,
    ContributionLevel.NONE: 0,
}

In [ ]:
# =============================================================================
# Feature Knowledge Base
# =============================================================================

@dataclass
class FeatureKnowledge:
    key: str
    title: str
    business_meaning: str
    direction: Direction
    weight: float          # relative importance for ranking root causes only
    format_kind: str       # "days" | "ratio" | "percent" | "units" | "currency"


# Replace the existing fmt_value function with this:
def fmt_value(value: float, kind: str) -> str:
    """Render a raw feature value as a business-friendly, conversational string."""
    if pd.isna(value):
        return "unavailable"

    # Helper for large numbers to "K" and "M"
    def human_number(v: float, is_currency: bool = False) -> str:
        prefix = "$" if is_currency else ""
        if v >= 1_000_000:
            return f"{prefix}{v/1_000_000:.1f}M"
        if v >= 1_000:
            return f"{prefix}{v/1_000:.0f}K"
        return f"{prefix}{v:,.0f}" if is_currency else f"{v:,.0f}"

    if kind == "days":
        days = int(round(value))
        if days <= 0: return "less than a day"
        if days == 1: return "about 1 day"
        return f"about {days} days"
    if kind == "ratio":
        return f"{value:.2f}x"
    if kind == "percent":
        return f"{value * 100:.0f}%"
    if kind == "units":
        return human_number(value, is_currency=False)
    if kind == "currency":
        return human_number(value, is_currency=True)
    return f"{value:,.2f}"

# =============================================================================
# Total Financial Exposure — v2: expected-loss formulation
# =============================================================================
# v1 computed shortage_exposure + lead_time_exposure, treating them as two
# independent, additive dollar amounts. That double-counts: lead time
# exposure is PART OF WHY a shortage might occur, not a wholly separate
# risk sitting alongside it.
#
# v2 instead computes an expected loss:
#
#     Total Financial Exposure = P(stockout) x Cost of Stockout
#
# - Cost of Stockout reuses shortage_exposure as-is — it's already the
#   estimated dollar impact if the shortage isn't resolved, so there's
#   nothing to change there.
# - P(stockout) is estimated from the record's own stock-position
#   features (stock_sufficiency_ratio, safety_stock_ratio, demand_pressure)
#   by reusing REFERENCE_DISTRIBUTION — the same portfolio percentile
#   markers the contribution engine already builds in Section 7. No new
#   statistical infrastructure and no upstream data changes required.
# - lead_time_exposure no longer contributes its own separate dollar
#   figure. Instead, its percentile position in the portfolio becomes a
#   MULTIPLIER (up to 1.5x) on the base probability: a longer or more
#   volatile replenishment window makes an actual stockout more likely
#   at the same stock position, but it isn't independently-countable
#   dollars at risk on top of the shortage cost itself.
#
# This is a real behavior change to the exported numbers, not just a
# formula tweak — see the reasoning above before relying on it for a
# live decision.

def _percentile_position(value: float, dist: "FeatureDistribution") -> float:
    """Map a raw value to an approximate 0-1 percentile position within
    the portfolio, via linear interpolation between the five markers
    (p10, q1, median, q3, p90) already computed in REFERENCE_DISTRIBUTION."""
    if pd.isna(value):
        return 0.5  # unknown -> assume mid-portfolio, neither optimistic nor alarmist
    markers = [
        (dist.p10, 0.10), (dist.q1, 0.25), (dist.median, 0.50),
        (dist.q3, 0.75), (dist.p90, 0.90),
    ]
    if value <= markers[0][0]:
        return 0.10
    if value >= markers[-1][0]:
        return 0.90
    for (lo_val, lo_pct), (hi_val, hi_pct) in zip(markers, markers[1:]):
        if lo_val <= value <= hi_val:
            if hi_val == lo_val:
                return lo_pct
            frac = (value - lo_val) / (hi_val - lo_val)
            return lo_pct + frac * (hi_pct - lo_pct)
    return 0.50


def estimate_stockout_probability(record: pd.Series) -> Optional[float]:
    """Estimate P(stockout) in [0, 1] from stock-position features.

    stock_sufficiency_ratio and safety_stock_ratio are lower-is-riskier,
    so we use (1 - percentile position): a ratio near the portfolio's
    LOW end scores a HIGH probability contribution. demand_pressure is
    higher-is-riskier, used directly. lead_time_exposure is applied
    afterward as a multiplier, not averaged in as a fourth component —
    see compute_total_financial_exposure.

    Returns None if none of the three inputs are available."""
    components = []
    if "stock_sufficiency_ratio" in record.index and pd.notna(record["stock_sufficiency_ratio"]):
        components.append(1 - _percentile_position(
            record["stock_sufficiency_ratio"], REFERENCE_DISTRIBUTION["stock_sufficiency_ratio"]
        ))
    if "safety_stock_ratio" in record.index and pd.notna(record["safety_stock_ratio"]):
        components.append(1 - _percentile_position(
            record["safety_stock_ratio"], REFERENCE_DISTRIBUTION["safety_stock_ratio"]
        ))
    if "demand_pressure" in record.index and pd.notna(record["demand_pressure"]):
        components.append(_percentile_position(
            record["demand_pressure"], REFERENCE_DISTRIBUTION["demand_pressure"]
        ))

    if not components:
        return None

    base_probability = sum(components) / len(components)

    if "lead_time_exposure" in record.index and pd.notna(record["lead_time_exposure"]):
        lead_time_percentile = _percentile_position(
            record["lead_time_exposure"], REFERENCE_DISTRIBUTION["lead_time_exposure"]
        )
        lead_time_multiplier = 1.0 + (0.5 * lead_time_percentile)  # up to +50%
    else:
        lead_time_multiplier = 1.0

    return min(1.0, base_probability * lead_time_multiplier)


def compute_total_financial_exposure(record: pd.Series) -> Optional[float]:
    """Return the raw numeric Total Financial Exposure as an expected
    loss — P(stockout) x Cost of Stockout — or None if there's nothing
    to compute it from. Kept as a plain number, not a formatted string,
    so the exported column can still be summed, sorted, or charted
    downstream instead of being text that looks like a number but isn't.

    Falls back gracefully rather than silently dropping a record v1
    would have reported a value for:
    - shortage_exposure present but no probability inputs available ->
      report the raw cost of stockout, un-weighted.
    - shortage_exposure missing entirely but lead_time_exposure present
      -> fall back to v1's behavior for that one case, since there's no
      cost-of-stockout figure to build an expected loss from."""
    cost_of_stockout = record.get("shortage_exposure")
    if pd.notna(cost_of_stockout):
        probability = estimate_stockout_probability(record)
        if probability is not None:
            return float(cost_of_stockout) * float(probability)
        return float(cost_of_stockout)

    if "lead_time_exposure" in record.index and pd.notna(record["lead_time_exposure"]):
        return float(record["lead_time_exposure"])

    return None


def format_financial_exposure(total: Optional[float]) -> str:
    """Human-readable "$45,230" / "$1.2M" string — used ONLY for narrative
    text (the executive summary sentence), never for the exported column."""
    if total is None:
        return "not estimated"
    return f"${total:,.0f}" if total < 1_000_000 else f"${total/1_000_000:.1f}M"

ROOT_CAUSE_KNOWLEDGE: Dict[str, FeatureKnowledge] = {
    "inventory_coverage_days": FeatureKnowledge(
        key="inventory_coverage_days",
        title="Inventory Coverage",
        business_meaning=(
            "The number of days the currently available stock is expected to "
            "satisfy demand before replenishment is required."
        ),
        direction=Direction.LOWER_IS_RISKIER,
        weight=0.20,
        format_kind="days",
    ),
    "stock_sufficiency_ratio": FeatureKnowledge(
        key="stock_sufficiency_ratio",
        title="Stock Sufficiency",
        business_meaning=(
            "How available stock compares to the reorder threshold; values "
            "below 1.0x mean stock has already dropped under the point at "
            "which replenishment should be triggered."
        ),
        direction=Direction.LOWER_IS_RISKIER,
        weight=0.15,
        format_kind="ratio",
    ),
    "safety_stock_ratio": FeatureKnowledge(
        key="safety_stock_ratio",
        title="Safety Stock Buffer",
        business_meaning=(
            "Available stock relative to the safety stock buffer set aside to "
            "absorb demand or supply variability; values below 1.0x mean the "
            "protective buffer has already been eroded."
        ),
        direction=Direction.LOWER_IS_RISKIER,
        weight=0.15,
        format_kind="ratio",
    ),
    "allocation_ratio": FeatureKnowledge(
        key="allocation_ratio",
        title="Stock Allocation Pressure",
        business_meaning=(
            "The share of on-hand stock already committed to existing orders, "
            "leaving less free stock available to absorb new demand."
        ),
        direction=Direction.HIGHER_IS_RISKIER,
        weight=0.10,
        format_kind="percent",
    ),
    "demand_pressure": FeatureKnowledge(
        key="demand_pressure",
        title="Demand Pressure",
        business_meaning=(
            "Forecasted demand relative to available stock; values above 1.0x "
            "mean forecasted demand exceeds what is currently on hand."
        ),
        direction=Direction.HIGHER_IS_RISKIER,
        weight=0.10,
        format_kind="ratio",
    ),
    "lead_time_exposure": FeatureKnowledge(
        key="lead_time_exposure",
        title="Lead Time Exposure",
        business_meaning=(
            "The value tied up and put at risk while waiting for the supplier "
            "replenishment lead time to elapse; the longer or more variable "
            "the lead time, the more exposure this represents."
        ),
        direction=Direction.HIGHER_IS_RISKIER,
        weight=0.10,
        format_kind="currency",
    ),
    "stock_gap": FeatureKnowledge(
        key="stock_gap",
        title="Stock Gap",
        business_meaning=(
            "The shortfall between available stock and the reorder threshold; "
            "positive values indicate stock has fallen below the level at "
            "which replenishment should already be underway."
        ),
        direction=Direction.HIGHER_IS_RISKIER,
        weight=0.10,
        format_kind="units",
    ),
    "shortage_exposure": FeatureKnowledge(
        key="shortage_exposure",
        title="Shortage Exposure",
        business_meaning=(
            "The estimated financial exposure that would be incurred if the "
            "current shortage is not resolved before stock is depleted."
        ),
        direction=Direction.HIGHER_IS_RISKIER,
        weight=0.10,
        format_kind="currency",
    ),
}

assert set(ROOT_CAUSE_KNOWLEDGE) == set(ENGINEERED_FEATURES)
assert abs(sum(k.weight for k in ROOT_CAUSE_KNOWLEDGE.values()) - 1.0) < 1e-9

pd.DataFrame([
    {
        "Feature": k.title,
        "Business Meaning": k.business_meaning,
        "Direction": k.direction.value,
        "Weight": k.weight,
    }
    for k in ROOT_CAUSE_KNOWLEDGE.values()
])

,Feature,Business Meaning,Direction,Weight
0,Inventory Coverage,The number of days the currently available stock is expected to satisfy demand before replenishment is required.,lower_is_riskier,0.20
1,Stock Sufficiency,How available stock compares to the reorder threshold; values below 1.0x mean stock has already dropped under the po...,lower_is_riskier,0.15
2,Safety Stock Buffer,Available stock relative to the safety stock buffer set aside to absorb demand or supply variability; values below 1...,lower_is_riskier,0.15
3,Stock Allocation Pressure,"The share of on-hand stock already committed to existing orders, leaving less free stock available to absorb new dem...",higher_is_riskier,0.10
4,Demand Pressure,Forecasted demand relative to available stock; values above 1.0x mean forecasted demand exceeds what is currently on...,higher_is_riskier,0.10
5,Lead Time Exposure,The value tied up and put at risk while waiting for the supplier replenishment lead time to elapse; the longer or mo...,higher_is_riskier,0.10
6,Stock Gap,The shortfall between available stock and the reorder threshold; positive values indicate stock has fallen below the...,higher_is_riskier,0.10
7,Shortage Exposure,The estimated financial exposure that would be incurred if the current shortage is not resolved before stock is depl...,higher_is_riskier,0.10


In [ ]:
# =============================================================================
# Demonstration: v1 (additive) vs v2 (expected-loss) Total Financial Exposure
# =============================================================================
# Sanity-check the new formula against the old one on a sample of
# Medium/High/Critical records before trusting it for anything live.

def _v1_additive_exposure(record: pd.Series) -> Optional[float]:
    exposures = []
    if pd.notna(record.get("shortage_exposure")):
        exposures.append(record["shortage_exposure"])
    if pd.notna(record.get("lead_time_exposure")):
        exposures.append(record["lead_time_exposure"])
    return float(sum(exposures)) if exposures else None

_demo_sample = inventory_df[inventory_df["risk_level"].isin(["Critical", "High", "Medium"])].head(10)

pd.DataFrame([
    {
        "MPN": r.get("mpn"),
        "Risk Level": r.get("risk_level"),
        "Shortage Exposure": r.get("shortage_exposure"),
        "Lead Time Exposure": r.get("lead_time_exposure"),
        "Est. P(stockout)": estimate_stockout_probability(r),
        "v1 Total (additive)": _v1_additive_exposure(r),
        "v2 Total (expected loss)": compute_total_financial_exposure(r),
    }
    for _, r in _demo_sample.iterrows()
]).round(2)

,MPN,Risk Level,Shortage Exposure,Lead Time Exposure,Est. P(stockout),v1 Total (additive),v2 Total (expected loss)
0,LM384N/NOPB,Medium,97771.57,592106.40,1.00,689877.97,97771.57
1,MAX22191AUT+T,Medium,916.76,15843.60,0.94,16760.36,866.34
2,9FGL0251BKILF,Critical,110901.07,351620.40,1.00,462521.47,110901.07
3,OMAPL138EZWTA3,Medium,1610.58,45115.20,1.00,46725.78,1610.58
4,PI4IOE5V6416ZDEX,High,53686.30,689073.60,1.00,742759.90,53686.30
5,GS1674INTE3,High,456711.20,240733.20,1.00,697444.40,456711.20
6,PCA6408AHKX,Medium,15076.95,180438.00,1.00,195514.95,15076.95
7,SM768GE0B0000AB,Critical,127450.37,34579.20,0.75,162029.57,95614.53
8,SI52202A01BGM,High,126964.67,361594.80,0.99,488559.47,126037.57
9,MPC8248CVRTIEA,High,285895.08,36870.89,0.39,322765.97,110150.68


## 9. Business Language Library

This is the module that turns numbers into language a Supply Chain
Manager can act on without a data dictionary. Every feature has its own
narrative builder that produces, for a given value and contribution
level:

- **Technical Explanation** — a precise, metric-level statement (still
  free of statistical jargon).
- **Human Explanation** — a natural-language statement of what is
  actually happening, always referencing the record's real value.
- **Business Impact** — the operational consequence (stockout,
  emergency purchasing, delivery delay, procurement cost, etc.).
- **Recommendations** — practical next actions, calibrated to how
  severe the contribution is.

In [ ]:
# =============================================================================
# Feature Narrative Data Structure
# =============================================================================

@dataclass
class FeatureNarrative:
    technical: str
    human: str
    business_impact: str
    recommendations: List[str]


NarrativeBuilder = Callable[[float, "ContributionLevel"], FeatureNarrative]

In [ ]:
# =============================================================================
# Narrative Builder — Inventory Coverage
# =============================================================================

def build_coverage_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    days = f"{value:,.0f}"
    technical = (
        f"Inventory Coverage is currently {days} days, calculated from available "
        f"stock against the current consumption rate."
    )
    if contribution in (ContributionLevel.CRITICAL, ContributionLevel.HIGH):
        human = (
            f"At the current pace of use, this item will only last about {days} "
            f"more days. If the next delivery is even a little late, it will run "
            f"out before the new stock shows up."
        )
    elif contribution == ContributionLevel.MODERATE:
        human = (
            f"This item has about {days} days of stock left. That's not a lot of "
            f"cushion if the next delivery is delayed."
        )
    else:
        human = (
            f"This item has about {days} days of stock on hand, which is a "
            f"comfortable amount for now."
        )
    impact = {
        ContributionLevel.CRITICAL: "It's very likely this item runs out completely before the next delivery arrives, which could stop production or delay orders to customers.",
        ContributionLevel.HIGH: "There's a real chance this item runs out before the next delivery arrives.",
        ContributionLevel.MODERATE: "There isn't much room to absorb a late delivery or a sudden jump in demand.",
        ContributionLevel.MINOR: "The cushion here is a bit smaller than usual, but not urgent.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Expedite the next purchase order", "Review reorder point and lead time assumptions", "Consider an emergency transfer from another location"],
        ContributionLevel.HIGH: ["Expedite the next purchase order", "Review reorder point for this item"],
        ContributionLevel.MODERATE: ["Monitor coverage closely and confirm the next replenishment date"],
        ContributionLevel.MINOR: ["Monitor coverage as part of routine review"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Stock Sufficiency
# =============================================================================

def build_sufficiency_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    ratio = f"{value:.2f}x"
    pct = f"{value * 100:,.0f}%"
    technical = (
        f"Stock Sufficiency is currently {ratio}, comparing available stock "
        f"against the reorder point."
    )
    if value < 1.0:
        human = (
            f"Stock has already dropped below the point where a new order "
            f"should normally be placed — it's only at about {pct} of that "
            f"level, so a reorder is already overdue."
        )
    else:
        human = (
            f"Stock is still above the point where a new order would normally "
            f"need to be placed, so there's no immediate concern here."
        )
    impact = {
        ContributionLevel.CRITICAL: "Stock is far below the point where it should have already been reordered, so a shortage is very likely if nothing is done soon.",
        ContributionLevel.HIGH: "Stock has dropped well past the point where a new order should have gone in, making a near-term shortage likely.",
        ContributionLevel.MODERATE: "Stock has dipped under the reorder point, so it's worth keeping an eye on.",
        ContributionLevel.MINOR: "Stock is only just under the reorder point.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Trigger an immediate replenishment order", "Escalate to procurement for expedited handling"],
        ContributionLevel.HIGH: ["Trigger replenishment ahead of the standard schedule"],
        ContributionLevel.MODERATE: ["Review the reorder point against current demand"],
        ContributionLevel.MINOR: ["Continue routine replenishment monitoring"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Safety Stock Buffer
# =============================================================================

def build_safety_stock_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    ratio = f"{value:.2f}x"
    pct = f"{value * 100:,.0f}%"
    technical = (
        f"Safety Stock Ratio is currently {ratio}, comparing available stock "
        f"against the defined safety stock level."
    )
    if value < 1.0:
        human = (
            f"The extra stock kept on hand for emergencies is already running "
            f"low — only about {pct} of that safety cushion is left."
        )
    else:
        human = (
            f"The extra stock kept on hand for emergencies is still in good "
            f"shape for this item."
        )
    impact = {
        ContributionLevel.CRITICAL: "The safety cushion is basically gone, so there's no protection left if a delivery is late or demand jumps.",
        ContributionLevel.HIGH: "Most of the safety cushion has been used up, leaving little protection against surprises.",
        ContributionLevel.MODERATE: "The safety cushion is smaller than it should be.",
        ContributionLevel.MINOR: "The safety cushion is slightly smaller than usual.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Increase safety stock for this item", "Expedite purchase order to rebuild the buffer"],
        ContributionLevel.HIGH: ["Increase safety stock for this item", "Review supplier lead time reliability"],
        ContributionLevel.MODERATE: ["Monitor the safety stock buffer and rebuild opportunistically"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Stock Allocation Pressure
# =============================================================================

def build_allocation_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    pct = f"{value * 100:,.0f}%"
    technical = (
        f"Allocation Ratio is currently {pct}, representing the share of "
        f"on-hand stock already committed to existing orders."
    )
    human = (
        f"About {pct} of the stock sitting in the warehouse is already promised "
        f"to other orders, so it isn't really free to cover anything new."
    )
    impact = {
        ContributionLevel.CRITICAL: "Almost none of the stock on hand is actually free to use — it's essentially all spoken for already.",
        ContributionLevel.HIGH: "Most of the stock on hand is already spoken for, leaving little room for new orders.",
        ContributionLevel.MODERATE: "A good chunk of the stock on hand is already committed elsewhere.",
        ContributionLevel.MINOR: "A small portion of the stock on hand is already committed.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Review open allocations for orders that can be reprioritized", "Expedite replenishment to free up future stock"],
        ContributionLevel.HIGH: ["Review open allocations and confirm order priorities"],
        ContributionLevel.MODERATE: ["Monitor allocation levels as part of routine review"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Demand Pressure
# =============================================================================

def build_demand_pressure_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    ratio = f"{value:.2f}x"
    technical = (
        f"Demand Pressure is currently {ratio}, comparing forecasted demand "
        f"against available stock."
    )
    if value > 1.0:
        overage_pct = f"{(value - 1) * 100:,.0f}%"
        human = (
            f"People want more of this item than there currently is in stock — "
            f"forecasted demand is running about {overage_pct} higher than "
            f"what's available."
        )
    else:
        used_pct = f"{value * 100:,.0f}%"
        human = (
            f"Forecasted demand is using up about {used_pct} of what's "
            f"currently available, which still leaves some room."
        )
    impact = {
        ContributionLevel.CRITICAL: "Demand is far outpacing what's in stock, so running out is very likely without quick action.",
        ContributionLevel.HIGH: "Demand is clearly outpacing what's in stock, raising the chance of running out.",
        ContributionLevel.MODERATE: "Demand is close to catching up with what's available.",
        ContributionLevel.MINOR: "Demand is only slightly elevated compared to what's on hand.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Review the demand forecast for accuracy", "Expedite replenishment to close the gap", "Evaluate temporary allocation limits to key customers"],
        ContributionLevel.HIGH: ["Review the demand forecast for this item", "Expedite the next replenishment"],
        ContributionLevel.MODERATE: ["Monitor demand trends closely"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Lead Time Exposure
# =============================================================================

def build_lead_time_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    amount = f"${value:,.0f}"
    technical = (
        f"Lead Time Exposure is currently {amount}, reflecting value at risk "
        f"during the supplier replenishment window."
    )
    human = (
        f"About {amount} worth of value is riding on the supplier delivering "
        f"this order on time. If the delivery is late, that's the value left "
        f"exposed in the meantime."
    )
    impact = {
        ContributionLevel.CRITICAL: "A very large amount of value depends on this delivery arriving on time, with little room to absorb a delay.",
        ContributionLevel.HIGH: "A significant amount of value depends on this delivery arriving on time.",
        ContributionLevel.MODERATE: "A moderate amount of value depends on this delivery arriving as expected.",
        ContributionLevel.MINOR: "Only a small amount of value depends on timely delivery here.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Monitor the supplier closely for on-time delivery", "Evaluate a backup or secondary supplier", "Expedite the outstanding purchase order"],
        ContributionLevel.HIGH: ["Monitor the supplier for on-time delivery", "Evaluate a backup supplier for this item"],
        ContributionLevel.MODERATE: ["Track the purchase order status against the expected delivery date"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Stock Gap
# =============================================================================

def build_stock_gap_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    units = f"{value:,.0f}"
    technical = (
        f"Stock Gap is currently {units} units, measuring the shortfall "
        f"between available stock and the reorder threshold."
    )
    if value > 0:
        human = (
            f"There simply isn't enough stock on hand — it's short by about "
            f"{units} units compared to what it should have by now, which "
            f"raises the risk of running out."
        )
    else:
        human = (
            f"Stock is actually {abs(value):,.0f} units above what's needed "
            f"right now, so this item isn't contributing to a shortage."
        )
    impact = {
        ContributionLevel.CRITICAL: "The shortage is severe enough that running out — and needing an emergency order — is very likely.",
        ContributionLevel.HIGH: "This shortage is large enough to likely cause a stockout and extra rush-order costs.",
        ContributionLevel.MODERATE: "This shortage could cause problems if it isn't addressed soon.",
        ContributionLevel.MINOR: "This is a small, manageable shortfall.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Expedite purchase order to close the gap", "Consider emergency purchasing or inventory transfer"],
        ContributionLevel.HIGH: ["Expedite purchase order to close the gap", "Evaluate transferring stock from another warehouse"],
        ContributionLevel.MODERATE: ["Confirm the next replenishment closes the gap on schedule"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder — Shortage Exposure
# =============================================================================

def build_shortage_exposure_narrative(value: float, contribution: ContributionLevel) -> FeatureNarrative:
    amount = f"${value:,.0f}"
    technical = (
        f"Shortage Exposure is currently {amount}, estimating the financial "
        f"impact if the current shortage is not resolved."
    )
    human = (
        f"If this shortage isn't fixed before stock runs out, it could end up "
        f"costing around {amount} in lost sales or extra expenses."
    )
    impact = {
        ContributionLevel.CRITICAL: "This could get expensive fast if it isn't fixed soon — it needs immediate attention.",
        ContributionLevel.HIGH: "There's real money on the line here if this isn't addressed soon.",
        ContributionLevel.MODERATE: "There's some financial risk tied to this item's current position.",
        ContributionLevel.MINOR: "The financial risk here is small.",
        ContributionLevel.NONE: "This isn't a concern right now.",
    }[contribution]
    recommendations = {
        ContributionLevel.CRITICAL: ["Prioritize this item for expedited purchasing", "Escalate to procurement leadership"],
        ContributionLevel.HIGH: ["Prioritize this item in the next procurement cycle"],
        ContributionLevel.MODERATE: ["Track financial exposure as part of routine review"],
        ContributionLevel.MINOR: ["No immediate action required"],
        ContributionLevel.NONE: ["No action required"],
    }[contribution]
    return FeatureNarrative(technical, human, impact, recommendations)

In [ ]:
# =============================================================================
# Narrative Builder Registry
# =============================================================================

NARRATIVE_BUILDERS: Dict[str, NarrativeBuilder] = {
    "inventory_coverage_days": build_coverage_narrative,
    "stock_sufficiency_ratio": build_sufficiency_narrative,
    "safety_stock_ratio": build_safety_stock_narrative,
    "allocation_ratio": build_allocation_narrative,
    "demand_pressure": build_demand_pressure_narrative,
    "lead_time_exposure": build_lead_time_narrative,
    "stock_gap": build_stock_gap_narrative,
    "shortage_exposure": build_shortage_exposure_narrative,
}

assert set(NARRATIVE_BUILDERS) == set(ENGINEERED_FEATURES)
logger.info("Business Language Library ready — narrative builders registered for all features")

## 10. Root Cause Analysis Engine

The engine evaluates every feature for a record, decides how much it is
contributing to the current risk, and ranks contributors so the
strongest drivers surface first.

Contribution is graded by comparing the record's value against the
feature's position in the wider portfolio (using the internal reference
distribution from Section 7) together with the feature's risk direction
from the Knowledge Base. No new risk formula is introduced — this only
determines *which already-known risk driver matters most for this
specific record*.

In [ ]:
# =============================================================================
# Feature Contribution Detection
# =============================================================================

def evaluate_contribution(value: float, feature: str, record: pd.Series = None) -> ContributionLevel:
    """Grade how much a feature value is contributing to risk. Includes business threshold overrides."""
    if pd.isna(value):
        return ContributionLevel.NONE

    # --- BUSINESS THRESHOLD OVERRIDES (Rule-based logic adjustment) ---
    # If coverage is under 5 days, it is ALWAYS Critical, regardless of portfolio percentiles.
    if feature == "inventory_coverage_days" and value < 5:
        return ContributionLevel.CRITICAL
    # If stock gap is greater than 0, it is at least Moderate.
    if feature == "stock_gap" and value > 0:
        # Check if it's extreme enough for Critical/High based on percentiles, but ensure it's not NONE.
        if value > REFERENCE_DISTRIBUTION["stock_gap"].q3:
            return ContributionLevel.HIGH
        return ContributionLevel.MODERATE
    # If sufficiency is below 0.50 (50%), it is Critical.
    if feature == "stock_sufficiency_ratio" and value < 0.50:
        return ContributionLevel.CRITICAL
    # --- END OF OVERRIDES ---

    # Standard statistical percentile logic
    dist = REFERENCE_DISTRIBUTION[feature]
    direction = ROOT_CAUSE_KNOWLEDGE[feature].direction

    if direction == Direction.LOWER_IS_RISKIER:
        if value <= dist.p10: return ContributionLevel.CRITICAL
        if value <= dist.q1: return ContributionLevel.HIGH
        if value <= dist.median: return ContributionLevel.MODERATE
        if value <= dist.q3: return ContributionLevel.MINOR
        return ContributionLevel.NONE
    else:
        if value >= dist.p90: return ContributionLevel.CRITICAL
        if value >= dist.q3: return ContributionLevel.HIGH
        if value >= dist.median: return ContributionLevel.MODERATE
        if value >= dist.q1: return ContributionLevel.MINOR
        return ContributionLevel.NONE

In [ ]:
# =============================================================================
# Feature Contribution Result
# =============================================================================

@dataclass
class FeatureContribution:
    feature: str
    title: str
    raw_value: float
    display_value: str
    weight: float
    contribution: ContributionLevel
    severity_score: float  # weight x severity rank — used only to sort/select


def analyze_feature_contribution(record: pd.Series, feature: str) -> FeatureContribution:
    """Evaluate one engineered feature for one inventory record."""
    knowledge = ROOT_CAUSE_KNOWLEDGE[feature]
    value = record[feature]
    contribution = evaluate_contribution(value, feature, record)
    severity_score = knowledge.weight * _SEVERITY_RANK[contribution]
    return FeatureContribution(
        feature=feature,
        title=knowledge.title,
        raw_value=value,
        display_value=fmt_value(value, knowledge.format_kind),
        weight=knowledge.weight,
        contribution=contribution,
        severity_score=severity_score,
    )

def determine_priority(risk_level: str, root_causes: List[FeatureContribution]) -> str:
    """Return a simple action priority based on risk level and root cause severity."""
    if risk_level not in ("High", "Critical", "Medium"):
        return "Monitor"
    severe_count = sum(1 for c in root_causes if c.contribution in (ContributionLevel.CRITICAL, ContributionLevel.HIGH))
    if risk_level == "Critical":
        return "Immediate" if severe_count >= 1 else "High"
    if risk_level == "High":
        return "High" if severe_count >= 1 else "Medium"
    # risk_level == "Medium" — previously fell through to the generic
    # "Monitor" bucket even though Medium records are analyzed by the
    # engine; now gets its own tier so a Medium record with a genuinely
    # severe contributing feature isn't under-prioritized.
    return "Medium" if severe_count >= 1 else "Low"

def analyze_record_contributions(record: pd.Series) -> List[FeatureContribution]:
    """Evaluate all engineered features together for one inventory record."""
    return [analyze_feature_contribution(record, feature) for feature in ENGINEERED_FEATURES]

In [ ]:
# Demonstration: contribution detection for a single Critical-risk record
demo_record = inventory_df[inventory_df["risk_level"] == "Critical"].iloc[0]
demo_contributions = analyze_record_contributions(demo_record)

pd.DataFrame([
    {
        "Feature": c.title,
        "Current Value": c.display_value,
        "Contribution": c.contribution.value,
    }
    for c in sorted(demo_contributions, key=lambda c: -c.severity_score)
])

,Feature,Current Value,Contribution
0,Stock Sufficiency,0.10x,Critical Contributor
1,Safety Stock Buffer,0.13x,High Contributor
2,Inventory Coverage,about 48 days,Moderate Contributor
3,Shortage Exposure,$111K,Critical Contributor
4,Demand Pressure,0.89x,High Contributor
5,Lead Time Exposure,$352K,High Contributor
6,Stock Gap,31K,High Contributor
7,Stock Allocation Pressure,12%,No Contribution


## 11. Business Explanation Generator

This module selects which contributing features count as the record's
**root causes**, and combines their individual narratives into one
coherent Human Explanation and one combined Business Impact statement.

Root causes are, in priority order:

1. Every **Critical Contributor** and **High Contributor**.
2. If none exist, the strongest **Moderate Contributors**, so a report
   is never left empty for a record that is genuinely at risk.

The result is capped at `MAX_ROOT_CAUSES_PER_RECORD` so the explanation
stays focused on what matters most instead of listing every feature.

In [ ]:
# =============================================================================
# Root Cause Selection
# =============================================================================

def select_root_causes(contributions: List[FeatureContribution]) -> List[FeatureContribution]:
    """Pick the features that best explain the current risk for a record."""
    ranked = sorted(contributions, key=lambda c: -c.severity_score)
    causes = [
        c for c in ranked
        if c.contribution in (ContributionLevel.CRITICAL, ContributionLevel.HIGH)
    ]
    if not causes:
        causes = [c for c in ranked if c.contribution == ContributionLevel.MODERATE][:2]
    return causes[:MAX_ROOT_CAUSES_PER_RECORD]

In [ ]:
# =============================================================================
# Combine Narratives Into Human Explanation & Business Impact
# =============================================================================

def build_explanation_bundle(root_causes: List[FeatureContribution]):
    """Turn the selected root causes into one Human Explanation and one
    Business Impact statement, plus a de-duplicated recommendation list."""
    if not root_causes:
        return (
            "No individual inventory feature stands out as a significant "
            "driver of risk for this record based on the current data.",
            "No material operational impact identified.",
            ["Continue routine monitoring"],
        )

    narratives = [
        (cause, NARRATIVE_BUILDERS[cause.feature](cause.raw_value, cause.contribution))
        for cause in root_causes
    ]

    human_explanation = " ".join(n.human for _, n in narratives)
    business_impact = " ".join(n.business_impact for _, n in narratives)

    seen = set()
    recommendations: List[str] = []
    for _, n in narratives:
        for rec in n.recommendations:
            if rec != "No action required" and rec not in seen:
                seen.add(rec)
                recommendations.append(rec)
    if not recommendations:
        recommendations = ["Continue routine monitoring"]

    return human_explanation, business_impact, recommendations

## 12. Recommendation Engine

Recommendations are generated as part of each feature's narrative
(Section 9) and de-duplicated across the selected root causes in Section
11. This section exposes that behaviour as a standalone function so
recommendations can also be regenerated for an arbitrary set of
contributions — for example, if a downstream ERP workflow only wants the
single top recommendation.

In [ ]:
# =============================================================================
# Recommendation Engine
# =============================================================================

def generate_recommendations(root_causes: List[FeatureContribution], top_n: Optional[int] = None) -> List[str]:
    """Return the de-duplicated, priority-ordered recommendations for a
    set of root causes. Recommendations from higher-weighted, more severe
    contributors are kept first."""
    ordered_causes = sorted(root_causes, key=lambda c: -c.severity_score)
    _, _, recommendations = build_explanation_bundle(ordered_causes)
    return recommendations[:top_n] if top_n else recommendations

## 13. Executive Summary Generator

A short paragraph, suitable for a leadership dashboard or a daily risk
digest, summarizing the item, its risk level, and its primary drivers.

In [ ]:
# =============================================================================
# Executive Summary Generator
# =============================================================================

def generate_executive_summary(record: pd.Series, root_causes: List[FeatureContribution]) -> str:
    mpn = record.get("mpn", "This item")
    risk_level = record.get("risk_level", "elevated")

    if not root_causes:
        return (
            f"Item {mpn} is currently classified as {risk_level} risk. No single "
            f"inventory feature stands out as a dominant driver; continued "
            f"routine monitoring is recommended."
        )

    cause_titles = [c.title for c in root_causes]
    if len(cause_titles) == 1:
        cause_text = cause_titles[0]
    elif len(cause_titles) == 2:
        cause_text = f"{cause_titles[0]} and {cause_titles[1]}"
    else:
        cause_text = ", ".join(cause_titles[:-1]) + f", and {cause_titles[-1]}"

    return (
        f"Item {mpn} is currently classified as {risk_level} risk, primarily "
        f"driven by {cause_text}. Timely action on these areas is recommended "
        f"to reduce the likelihood of stockout, delivery delays, or additional "
        f"procurement cost."
    )

## 14. Inventory Record Analyzer

This is the orchestrator that ties every module above together for a
single inventory record, producing the final Root Cause Report:

```
Inventory Record
      ↓
Feature Evaluation           (Section 10)
      ↓
Feature Contribution Detection (Section 10)
      ↓
Business Language Engine     (Section 9)
      ↓
Root Cause Generator          (Section 11)
      ↓
Recommendation Generator      (Section 12)
      ↓
Executive Summary             (Section 13)
```

In [ ]:
# =============================================================================
# Root Cause Report
# =============================================================================

@dataclass
class RootCauseReport:
    inventory_id: object
    mpn: object
    risk_score: float
    risk_level: str
    detected_root_causes: List[str]
    human_explanation: str
    business_impact: str
    executive_summary: str
    recommendations: List[str]
    validation_warnings: List[str] = field(default_factory=list)
    priority: str = "Monitor"                # NEW
    total_financial_exposure: Optional[float] = None  # NEW — numeric, not a formatted string


def generate_root_cause_report(record: pd.Series) -> RootCauseReport:
    """Run the full Root Cause Analysis pipeline for a single inventory
    record and return a structured report."""

    # --- Validation (unchanged) ---
    warnings_found = validate_record(record)
    if warnings_found:
        for w in warnings_found:
            logger.warning(f"[{record.get('inventory_id', 'unknown')}] {w}")

    # --- Analyze contributions (unchanged) ---
    contributions = analyze_record_contributions(record)
    root_causes = select_root_causes(contributions)

    # --- Build narratives (unchanged) ---
    human_explanation, business_impact, recommendations = build_explanation_bundle(root_causes)
    executive_summary = generate_executive_summary(record, root_causes)

    # --- NEW: Priority and Financial Exposure ---
    priority = determine_priority(record.get("risk_level", ""), root_causes)
    total_financial = compute_total_financial_exposure(record)  # numeric (float) or None

    # Append financial exposure to executive summary if enabled — the
    # narrative sentence still gets the friendly "$1.2M" formatting,
    # via format_financial_exposure(); only the *stored* value on the
    # report/CSV stays a plain number.
    if total_financial is not None and INCLUDE_FINANCIAL_IMPACT:
        executive_summary += f" Total estimated financial exposure: {format_financial_exposure(total_financial)}."

    # --- Return the complete report ---
    return RootCauseReport(
        inventory_id=record.get("inventory_id"),
        mpn=record.get("mpn"),
        risk_score=record.get("risk_score"),
        risk_level=record.get("risk_level"),
        detected_root_causes=[c.title for c in root_causes],
        human_explanation=human_explanation,
        business_impact=business_impact,
        executive_summary=executive_summary,
        recommendations=recommendations,
        validation_warnings=warnings_found,
        priority=priority,
        total_financial_exposure=total_financial,
    )


def print_root_cause_report(report: RootCauseReport) -> None:
    """Pretty-print a Root Cause Report in the format requested for
    downstream / ERP consumption."""
    print("=" * 78)
    print(f"Inventory ID : {report.inventory_id}")
    print(f"MPN          : {report.mpn}")
    print(f"Risk Score   : {report.risk_score:.2f}")
    print(f"Risk Level   : {report.risk_level}")
    print("-" * 78)
    print("Detected Root Causes")
    print("-" * 78)
    for cause in report.detected_root_causes:
        print(f"  - {cause}")
    print("-" * 78)
    print("Human Explanation")
    print("-" * 78)
    print(report.human_explanation)
    print("-" * 78)
    print("Business Impact")
    print("-" * 78)
    print(report.business_impact)
    print("-" * 78)
    print("Executive Summary")
    print("-" * 78)
    print(report.executive_summary)
    print("-" * 78)
    print("Recommended Actions")
    print("-" * 78)
    for rec in report.recommendations:
        print(f"  - {rec}")
    print("=" * 78)

## 15. Validation Layer

Every record is validated before an explanation is generated. Problems
never crash the notebook — they are logged as warnings and attached to
the record's report so they remain visible and auditable.

In [ ]:
# =============================================================================
# Record-Level Validation
# =============================================================================

def validate_record(record: pd.Series) -> List[str]:
    """Check one inventory record for missing values, invalid values, and
    unexpected ranges. Returns a list of warning strings; never raises."""
    warnings_found: List[str] = []

    for col in REQUIRED_COLUMNS:
        if col not in record.index:
            warnings_found.append(f"Missing column: {col}")
            continue
        if pd.isna(record[col]):
            warnings_found.append(f"Null value in {col}")

    for feature in ENGINEERED_FEATURES:
        if feature not in record.index or pd.isna(record[feature]):
            continue
        value = record[feature]
        if not np.isfinite(value):
            warnings_found.append(f"Non-finite value in {feature}")

    if "allocation_ratio" in record.index and pd.notna(record["allocation_ratio"]):
        if not (-0.01 <= record["allocation_ratio"] <= 1.5):
            warnings_found.append(
                f"Allocation ratio outside expected range: {record['allocation_ratio']}"
            )

    if "risk_level" in record.index and record["risk_level"] not in KNOWN_RISK_LEVELS:
        warnings_found.append(f"Unexpected risk_level value: {record['risk_level']}")

    return warnings_found

In [ ]:
# Demonstration: full single-record report for a Critical-risk item
demo_report = generate_root_cause_report(demo_record)
print_root_cause_report(demo_report)

Inventory ID : INV00008
MPN          : 9FGL0251BKILF
Risk Score   : 62.17
Risk Level   : Critical
------------------------------------------------------------------------------
Detected Root Causes
------------------------------------------------------------------------------
  - Stock Sufficiency
  - Safety Stock Buffer
  - Shortage Exposure
  - Demand Pressure
------------------------------------------------------------------------------
Human Explanation
------------------------------------------------------------------------------
Stock has already dropped below the point where a new order should normally be placed — it's only at about 10% of that level, so a reorder is already overdue. The extra stock kept on hand for emergencies is already running low — only about 13% of that safety cushion is left. If this shortage isn't fixed before stock runs out, it could end up costing around $110,901 in lost sales or extra expenses. Forecasted demand is using up about 89% of what's currentl

## 16. Batch Processing

The engine is now run across every Medium, High, and Critical risk
record in the dataset. Each record is processed independently and defensively — a
failure on one record is logged and skipped rather than stopping the
batch.

In [ ]:
# =============================================================================
# Batch Root Cause Analysis
# =============================================================================

def run_batch_root_cause_analysis(data: pd.DataFrame, risk_levels: List[str]) -> pd.DataFrame:
    """Run the Root Cause Analysis Engine over every record in the given
    risk levels and return a tidy results DataFrame."""

    target = data[data["risk_level"].isin(risk_levels)]
    logger.info(f"Running Root Cause Analysis on {len(target):,} records "
                f"(risk levels: {risk_levels})")

    rows = []
    error_count = 0

    for idx, record in target.iterrows():
        try:
            report = generate_root_cause_report(record)

            # Format recommendations as a numbered bullet list
            if report.recommendations:
                rec_str = "\n".join(f"{i+1}. {rec}" for i, rec in enumerate(report.recommendations))
            else:
                rec_str = "No specific action required"

            rows.append({
                "original_index": record.name, # Store the original index
                "Inventory ID": report.inventory_id,
                "MPN": report.mpn,
                "Risk Score": report.risk_score,
                "Risk Level": report.risk_level,
                "Priority": report.priority,
                "Inventory Date": record.get(INVENTORY_DATE_COLUMN, "")
                                   if INVENTORY_DATE_COLUMN in record.index else "",
                "Detected Root Causes": "; ".join(report.detected_root_causes),
                "Human Explanation": report.human_explanation,
                "Business Impact": report.business_impact,
                "Executive Summary": report.executive_summary,
                "Total Financial Exposure": report.total_financial_exposure,
                "Recommendations": rec_str,
            })

        except Exception as exc:
            error_count += 1
            logger.error(f"Failed to analyze record at index {idx}: {exc}")

    logger.info(f"Batch complete: {len(rows):,} succeeded, {error_count:,} failed")
    result_df = pd.DataFrame(rows)
    result_df.set_index("original_index", inplace=True) # Set the original index
    result_df.index.name = None # Remove the index name for cleaner display
    return result_df

In [ ]:
target_df = inventory_df[inventory_df["risk_level"].isin(HIGH_RISK_LEVELS)].copy()
root_cause_results = run_batch_root_cause_analysis(target_df, HIGH_RISK_LEVELS)

# =============================================================================
# Validation: Compare Engine Results vs Simple Business Heuristic
# =============================================================================

def validate_against_heuristic(record: pd.Series) -> bool:
    """Returns True if the engine's top root cause matches a simple heuristic."""
    # Simple heuristic: If coverage < 15 days, it MUST be a top cause.
    if record["inventory_coverage_days"] < 15:
        return "Inventory Coverage" in root_cause_results.loc[record.name, "Detected Root Causes"]

    # If stock gap > 0, it MUST be a top cause.
    if record["stock_gap"] > 0:
        return "Stock Gap" in root_cause_results.loc[record.name, "Detected Root Causes"]

    # If safety stock < 0.5x, it MUST be a top cause.
    if record["safety_stock_ratio"] < 0.5:
        return "Safety Stock Buffer" in root_cause_results.loc[record.name, "Detected Root Causes"]

    return True # passes if no extreme conditions are met

# Run validation on a sample of High/Critical records
validation_results = []
for idx, row in target_df.sample(min(50, len(target_df))).iterrows():
    is_correct = validate_against_heuristic(row)
    validation_results.append({
        "Inventory ID": row["inventory_id"],
        "MPN": row["mpn"],
        "Risk Level": row["risk_level"],
        "Top Root Cause(s)": root_cause_results.loc[row.name, "Detected Root Causes"],
        "Heuristic Passed": is_correct
    })

validation_df = pd.DataFrame(validation_results)
print(f"Validation Accuracy: {(validation_df['Heuristic Passed'].mean() * 100):.2f}%")
display(validation_df.head(10))

Validation Accuracy: 50.00%


,Inventory ID,MPN,Risk Level,Top Root Cause(s),Heuristic Passed
0,INV00823,R7S921052VCBG#BC0,High,Stock Sufficiency; Shortage Exposure,False
1,INV00214,LED173048RSLR,Critical,Inventory Coverage; Stock Sufficiency; Safety Stock Buffer; Demand Pressure,True
2,INV08145,AM3352BZCZ100,Medium,Inventory Coverage; Stock Sufficiency; Safety Stock Buffer; Stock Allocation Pressure,True
3,INV02350,SN65DPHY440SSRHRR,Medium,Inventory Coverage; Stock Sufficiency; Safety Stock Buffer; Demand Pressure,False
4,INV04259,PIC16F15223TI/SL,Medium,Inventory Coverage; Stock Sufficiency; Safety Stock Buffer; Lead Time Exposure,False
5,INV02300,R7S921047VCBG#BC0,High,Stock Sufficiency; Shortage Exposure,False
6,INV01402,DS28E15Q+T,Medium,Stock Sufficiency; Safety Stock Buffer; Stock Gap,True
7,INV00359,MCP3004I/SL,Medium,Inventory Coverage; Stock Sufficiency; Safety Stock Buffer; Lead Time Exposure,False
8,INV01642,ADN2814ACPZ,High,Stock Sufficiency; Lead Time Exposure; Shortage Exposure; Stock Gap,True
9,INV07030,DS2401P+T&R,Medium,Inventory Coverage; Stock Sufficiency; Safety Stock Buffer; Stock Allocation Pressure,True


In [ ]:
# Quick sanity check on batch coverage
print(f"Total Medium/High/Critical records : {(inventory_df['risk_level'].isin(HIGH_RISK_LEVELS)).sum():,}")
print(f"Records with a report        : {len(root_cause_results):,}")
print()
print("Most frequent root causes across the portfolio:")
(
    root_cause_results["Detected Root Causes"]
    .str.split("; ")
    .explode()
    .value_counts()
)

Total Medium/High/Critical records : 1,875
Records with a report        : 1,875

Most frequent root causes across the portfolio:


,count
Detected Root Causes,
Stock Sufficiency,1835
Safety Stock Buffer,1296
Inventory Coverage,984
Stock Allocation Pressure,615
Demand Pressure,568
Shortage Exposure,512
Lead Time Exposure,433
Stock Gap,412


## 16b. Threshold Validation Against Real Outcomes *(inactive — no ground-truth data yet)*

Everything above grades contribution and risk by comparing a record's
value against the rest of the **current** portfolio (percentiles) plus
a few hardcoded business overrides. That's internally consistent, but
it has never been checked against what actually happened — whether an
item flagged Critical really did stock out, or whether the root cause
this engine names was the real cause.

Doing that requires a ground-truth outcome column this dataset doesn't
have yet — something like `actual_stockout_occurred` (bool) or
`actual_stockout_date`, populated after the fact from real order/fulfillment
history. The cell below is a scaffold, not a working validation: it
checks for that column and, if absent, prints what it needs and skips
cleanly rather than pretending to validate against data that isn't there.
Once that column exists upstream, this becomes a real precision/recall
check on the CRITICAL threshold instead of a no-op.

In [ ]:
# =============================================================================
# Threshold Validation Against Real Outcomes (scaffold — see markdown above)
# =============================================================================

OUTCOME_COLUMN = "actual_stockout_occurred"  # update if your real column is named differently

if OUTCOME_COLUMN not in inventory_df.columns:
    print(
        f"⏭️  Skipping outcome validation — '{OUTCOME_COLUMN}' isn't in the dataset.\n"
        f"    This notebook has no way to know whether a Critical-flagged item\n"
        f"    actually stocked out without that ground truth. Add a boolean\n"
        f"    column recording the real outcome (from order/fulfillment history)\n"
        f"    and re-run this cell — it will then report precision/recall for\n"
        f"    the Critical threshold instead of skipping."
    )
else:
    # Real validation, only runs once the ground-truth column exists.
    validation_df = inventory_df[["risk_level", OUTCOME_COLUMN]].dropna()
    predicted_critical = validation_df["risk_level"] == "Critical"
    actual_stockout = validation_df[OUTCOME_COLUMN].astype(bool)

    true_positives = (predicted_critical & actual_stockout).sum()
    false_positives = (predicted_critical & ~actual_stockout).sum()
    false_negatives = (~predicted_critical & actual_stockout).sum()

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) else float("nan")
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) else float("nan")

    print(f"Critical-flag precision: {precision:.1%}  (of items flagged Critical, how many actually stocked out)")
    print(f"Critical-flag recall:    {recall:.1%}  (of items that actually stocked out, how many were flagged Critical)")
    print()
    print("If precision is low: the Critical threshold (currently p10/hardcoded overrides) is too loose — tighten it.")
    print("If recall is low: it's too strict and missing real stockouts — loosen it or add a missed risk driver.")

⏭️  Skipping outcome validation — 'actual_stockout_occurred' isn't in the dataset.
    This notebook has no way to know whether a Critical-flagged item
    actually stocked out without that ground truth. Add a boolean
    column recording the real outcome (from order/fulfillment history)
    and re-run this cell — it will then report precision/recall for
    the Critical threshold instead of skipping.


In [ ]:
root_cause_results = run_batch_root_cause_analysis(inventory_df, HIGH_RISK_LEVELS)

Cell 1: Ensure requests is installed (optional but safe)


# 17.LLM Polishing (Optional - Add at the end as a new cell)

Cell 2: Load your OpenRouter API key from Colab secrets


In [ ]:
# Cell 2: Load API key from Colab secrets
from google.colab import userdata
import requests
import pandas as pd

try:
    API_KEY = userdata.get('GROQ_API_KEY')
    if not API_KEY:
        raise ValueError("API key not found in secrets.")
    print("✅ API key loaded successfully.")
except Exception as e:
    print(f"❌ Error loading API key: {e}")
    API_KEY = None

✅ API key loaded successfully.


Cell 3: Test if the API key is valid (diagnostic)


In [ ]:
# Cell 3: Test API key validity
if API_KEY:
    url = "https://openrouter.ai/api/v1/models"
    headers = {"Authorization": f"Bearer {API_KEY}"}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            print("✅ API key is valid.")
        else:
            print(f"❌ API key rejected: {resp.status_code} - {resp.text}")
    except Exception as e:
        print(f"❌ Error testing API: {e}")
else:
    print("Skipping test – no API key.")

✅ API key is valid.


Cell 5: Main LLM Polishing and Saving

In [ ]:
!pip install -q groq

In [ ]:
!pip install -q google-genai

In [ ]:
from groq import Groq
API_KEY = userdata.get("GROQ_API_KEY")
if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in secrets.")
client = Groq(api_key=API_KEY)

In [ ]:
# IMPORTANT: LLM_MODEL must match the client instantiated in the cell
# below (currently anthropic.Anthropic()). If you swap in a different
# provider's model name (e.g. a Groq or OpenAI model like
# "llama-3.3-70b-versatile"), you must also swap the client instantiation
# a few cells down to that provider's SDK -- otherwise every call fails
# and safe_llm_polish() will silently fall back to the deterministic text
# for every record without any visible error beyond a logged warning.
POLISHED_EXPORT_FILE = OUTPUT_DIR / "inventory_root_cause_analysis_polished.csv"

In [ ]:
import os

CHECKPOINT_FILE = "merged_checkpoint.csv"
SAVE_EVERY = 10

In [ ]:
import google.genai
print(google.genai.__version__)

2.11.0


In [ ]:
FORCE_REPOLISH = True   # set to True to ignore existing checkpoint
if FORCE_REPOLISH and os.path.exists("merged_checkpoint.csv"):
    os.remove("merged_checkpoint.csv")

In [ ]:
import time
import re
import os
import pandas as pd
import traceback
from google.colab import userdata
from groq import Groq

# ----- Groq setup -----
API_KEY = userdata.get("GROQ_API_KEY")
if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in secrets.")
client = Groq(api_key=API_KEY)
# Per YOUR account's actual limits (console.groq.com/settings/limits):
#   llama-3.3-70b-versatile:  100K tokens/day, 12K tokens/minute
#   llama-3.1-8b-instant:     500K tokens/day,  6K tokens/minute  <- 5x the daily budget
# At ~568 tokens/row measured on the 70B model, 100K/day meant ~11 days
# to finish all ~1,875 rows. 500K/day cuts that to ~2 days instead,
# which is the actual fix -- switching model, not just batching harder.
# groq/compound has NO daily token limit at all on your account, but
# it's Groq's agentic/tool-use model family (web search + code
# execution), not a plain instruct model -- not swapping to it blind
# without knowing how it behaves on a simple text-rewrite task.
MODEL = "llama-3.1-8b-instant"   # 500K tokens/day on your account

# IMPORTANT: this model's per-minute cap (6K TPM) is HALF the 70B
# model's (12K TPM). A batch of 20 rows' max possible output alone
# (280*20+200 ~= 5,800 tokens) would nearly exhaust an entire minute's
# budget by itself, before even counting the input prompt tokens or
# leaving room for the next request. Sized down to 8 rows/batch instead
# to stay comfortably under 6K TPM per request -- if you switch MODEL
# back to a model with a higher TPM, this can go back up.
ROWS_PER_BATCH = 8
REQUEST_DELAY_SECONDS = 5.0
MAX_RETRIES = 5
RATE_LIMIT_BACKOFF_BASE = 5
RATE_LIMIT_BACKOFF_STEP = 20

CHECKPOINT_FILE = "merged_checkpoint.csv"
POLISHED_EXPORT_FILE = OUTPUT_DIR / "inventory_root_cause_analysis_polished.csv"
FORCE_REPOLISH = True   # set to True to re-polish from scratch
if FORCE_REPOLISH and os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

class DailyQuotaExhausted(Exception):
    pass

SECTION_NAMES = ["Human Explanation", "Business Impact", "Executive Summary"]

def build_batch_prompt(rows: list) -> str:
    blocks = []
    for i, row in enumerate(rows, start=1):
        blocks.append(
            f"=== Row {i} ===\n"
            f"Human Explanation:\n{row['Human Explanation']}\n\n"
            f"Business Impact:\n{row['Business Impact']}\n\n"
            f"Executive Summary:\n{row['Executive Summary']}\n"
        )
    joined = "\n".join(blocks)
    return f"""
You are a senior supply chain risk communication editor.

Below are {len(rows)} separate reports, each labeled "=== Row N ===".
Improve the writing of each report's three sections without changing
their meaning.

Requirements: Preserve every fact, number, score, and risk level exactly
as provided; never invent, remove, or infer information. Improve clarity,
grammar, flow, and business tone for procurement/supply chain managers;
avoid repeating information across sections; keep each section concise.
Rewrite any mention of an internal scoring policy (e.g. "hard-stop rule")
as "the project's risk policy" rather than exposing implementation
details. No Markdown, bullets, or intros. Return EVERY row, in order,
using exactly this format for each:

=== Row N ===
Human Explanation:
<rewritten text>

Business Impact:
<rewritten text>

Executive Summary:
<rewritten text>

Original Reports

{joined}
"""

def parse_batch_response(output: str) -> dict:
    results = {}
    row_pattern = re.compile(r"===\s*Row\s+(\d+)\s*===", re.IGNORECASE)
    matches = list(row_pattern.finditer(output))
    for m_idx, match in enumerate(matches):
        row_num = int(match.group(1))
        start = match.end()
        end = matches[m_idx + 1].start() if m_idx + 1 < len(matches) else len(output)
        block = output[start:end].strip()
        if not block:
            continue
        parsed = {}
        for section in SECTION_NAMES:
            header_pattern = re.compile(
                rf"{section}\s*[:–—-]?\s*(.*?)(?=\n\s*(?:{'|'.join(SECTION_NAMES)})\s*[:–—-]?|$)",
                re.DOTALL | re.IGNORECASE
            )
            match_section = header_pattern.search(block)
            if match_section:
                content = match_section.group(1).strip()
                for other in SECTION_NAMES:
                    if other != section:
                        idx = content.lower().find(other.lower())
                        if idx != -1 and (idx == 0 or content[idx-1] in ('\n', ' ', '\t')):
                            content = content[:idx].strip()
                            break
                parsed[section] = content
        if len(parsed) == len(SECTION_NAMES):
            results[row_num] = parsed
    return results

def polish_batch(rows: list) -> dict:
    prompt = build_batch_prompt(rows)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a senior supply chain communication editor. Improve writing only. Never change facts or numbers. Return every row, in order, in the requested format."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2,
                max_tokens=280 * len(rows) + 200,  # was 400/row; trimmed ceiling, not typical usage
            )
            output = response.choices[0].message.content.strip()
            return parse_batch_response(output)
        except Exception as e:
            print("=" * 80)
            traceback.print_exc()
            print("=" * 80)
            if "429" in str(e):
                error_text = str(e)
                if re.search(r"per[_ ]?day", error_text, re.IGNORECASE):
                    print("Daily quota exhausted for this API key/model.")
                    print("Waiting will not help - it only resets at midnight Pacific Time.")
                    raise DailyQuotaExhausted(error_text)
                suggested_match = re.search(
                    r"(?:retry|try\s*again|wait)\s*(?:in|after|for)?\s*(\d+(?:\.\d+)?)\s*(?:s|sec|second|seconds)?",
                    error_text,
                    re.IGNORECASE
                )
                suggested_wait = float(suggested_match.group(1)) if suggested_match else 0.0
                if suggested_match is None:
                    print("⚠️ Could not parse retry time from error. Using backoff only.")
                wait_time = max(5.0, suggested_wait) + RATE_LIMIT_BACKOFF_BASE + (RATE_LIMIT_BACKOFF_STEP * attempt)
                if attempt == MAX_RETRIES:
                    print(f"Rate limit reached. Giving up on this batch after {MAX_RETRIES} attempts.")
                    break
                print(f"Rate limit reached (attempt {attempt}/{MAX_RETRIES}). Waiting {wait_time:.0f} seconds...")
                time.sleep(wait_time)
                continue
            print(f"Non‑rate‑limit error: {e}")
            if attempt == MAX_RETRIES:
                break
            time.sleep(REQUEST_DELAY_SECONDS * attempt)
    return {}

# ---- Merging and polishing ----
if os.path.exists(CHECKPOINT_FILE):
    print(f"Loading checkpoint: {CHECKPOINT_FILE}")
    merged = pd.read_csv(CHECKPOINT_FILE)
else:
    merged = root_cause_results.merge(
        inventory_df[["inventory_id", "mpn", "shortage_exposure", "lead_time_exposure"]],
        left_on=["Inventory ID", "MPN"],
        right_on=["inventory_id", "mpn"],
        how="left"
    )
    merged["Total Financial Exposure"] = merged.apply(
        lambda r: compute_total_financial_exposure(r),
        axis=1
    )
    merged.drop(
        columns=["inventory_id", "mpn", "shortage_exposure", "lead_time_exposure"],
        inplace=True,
        errors="ignore"
    )
    merged["LLM_Polished"] = False   # mark as unpolished for fresh run

# Process eligible rows
SAMPLE_SIZE = None   # set to a small number for testing
eligible_mask = (
    merged["Risk Level"].isin(["High", "Critical", "Medium"])
    & (~merged["LLM_Polished"])
)
rows_to_process = merged.loc[eligible_mask]
if SAMPLE_SIZE is not None:
    rows_to_process = rows_to_process.head(SAMPLE_SIZE)

polished_count = 0
fallback_count = 0
total = len(rows_to_process)
stopped_early = False
indices = list(rows_to_process.index)

for batch_start in range(0, total, ROWS_PER_BATCH):
    batch_indices = indices[batch_start: batch_start + ROWS_PER_BATCH]
    batch_rows = [rows_to_process.loc[idx] for idx in batch_indices]
    try:
        results = polish_batch(batch_rows)
    except DailyQuotaExhausted:
        stopped_early = True
        break
    for pos, idx in enumerate(batch_indices, start=1):
        polished = results.get(pos)
        if polished:
            merged.at[idx, "Human Explanation"] = polished["Human Explanation"]
            merged.at[idx, "Business Impact"] = polished["Business Impact"]
            merged.at[idx, "Executive Summary"] = polished["Executive Summary"]
            merged.at[idx, "LLM_Polished"] = True
            polished_count += 1
        else:
            fallback_count += 1
    merged.to_csv(CHECKPOINT_FILE, index=False)
    done = min(batch_start + ROWS_PER_BATCH, total)
    print(f"   [{done}/{total}] polished={polished_count}  fell back to template={fallback_count}")
    if batch_start + ROWS_PER_BATCH < total:
        time.sleep(REQUEST_DELAY_SECONDS)

merged.to_csv(CHECKPOINT_FILE, index=False)

print()
if stopped_early:
    remaining = total - (polished_count + fallback_count)
    print(f"⏸️  Stopped early: daily quota exhausted after processing "
          f"{polished_count + fallback_count}/{total} rows ({remaining} rows remaining).")
    print(f"   Progress saved to {CHECKPOINT_FILE} - re-run after quota resets.")
else:
    print(f"✅ LLM polishing complete: {polished_count}/{total} rows actually polished, "
          f"{fallback_count}/{total} kept the deterministic template text.")

merged.to_csv(POLISHED_EXPORT_FILE, index=False)
print(f"✅ Polished CSV saved to: {POLISHED_EXPORT_FILE}")

   [8/1875] polished=8  fell back to template=0
   [16/1875] polished=16  fell back to template=0
   [24/1875] polished=24  fell back to template=0
   [32/1875] polished=32  fell back to template=0
   [40/1875] polished=40  fell back to template=0
   [48/1875] polished=48  fell back to template=0
   [56/1875] polished=56  fell back to template=0
   [64/1875] polished=64  fell back to template=0
   [72/1875] polished=72  fell back to template=0
   [80/1875] polished=80  fell back to template=0
   [88/1875] polished=88  fell back to template=0
   [96/1875] polished=96  fell back to template=0
   [104/1875] polished=104  fell back to template=0
   [112/1875] polished=112  fell back to template=0
   [120/1875] polished=120  fell back to template=0
   [128/1875] polished=128  fell back to template=0
   [136/1875] polished=136  fell back to template=0
   [144/1875] polished=144  fell back to template=0
   [152/1875] polished=152  fell back to template=0
   [160/1875] polished=160  fell back 

Traceback (most recent call last):
  File "/tmp/ipykernel_1476/1040071084.py", line 126, in polish_batch
    response = client.chat.completions.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/groq/resources/chat/completions.py", line 462, in create
    return self._post(
           ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/groq/_base_client.py", line 1284, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/groq/_base_client.py", line 1071, in request
    raise self._make_status_error_from_response(err.response) from None
groq.RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kys0yqn2efhv3p39bx5pkhbr` service tier `on_demand` on tokens per minute

In [ ]:
from google.colab import userdata

print(userdata.get("GEMINI_API_KEY"))

## 17. CSV Export

The final Root Cause Analysis results are exported to
`inventory_root_cause_analysis.csv` for downstream consumption by an
ERP system, BI dashboard, or a planning team's daily review.

In [ ]:
# =============================================================================
# Export Root Cause Analysis Results
# =============================================================================

root_cause_results.to_csv(EXPORT_FILE, index=False)
logger.info(f"Exported {len(root_cause_results):,} Root Cause Analysis reports to '{EXPORT_FILE}'")

root_cause_results.shape

In [ ]:
df = pd.read_csv(POLISHED_EXPORT_FILE, encoding="utf-8")
# Coerce to numeric rather than forcing to string — CSVs round-trip
# numbers as text anyway, but pd.to_numeric ensures any stray formatting
# (stray whitespace, an accidental "$", etc.) from the read-back doesn't
# silently turn this back into a non-numeric column. errors="coerce"
# turns anything unparseable into NaN instead of raising.
df["Total Financial Exposure"] = pd.to_numeric(df["Total Financial Exposure"], errors="coerce")
df.to_csv(POLISHED_EXPORT_FILE_new, index=False, encoding="utf-8-sig")

## Summary

- **Input**: the already-scored inventory dataset (Risk Score, Risk
  Level, and eight engineered features), reused exactly as produced by
  the upstream Inventory Risk Assessment notebook.
- **Engine**: a modular Root Cause Analysis pipeline — Feature
  Evaluation → Contribution Detection → Business Language Engine → Root
  Cause Generator → Recommendation Generator → Executive Summary.
- **Output**: `outputs/inventory_root_cause_analysis.csv`, one row per
  Medium/High/Critical risk record, with Inventory ID, MPN, Risk Score, Risk
  Level, Detected Root Causes, Human Explanation, Business Impact,
  Executive Summary, and Recommendations — all written in business
  language, free of statistical terminology.
- **Total Financial Exposure (v2)**: changed from an additive
  `shortage_exposure + lead_time_exposure` to an expected-loss formula,
  `P(stockout) x Cost of Stockout`. Lead time exposure now raises the
  estimated stockout probability instead of contributing its own
  separate dollar figure, avoiding the double-counting the additive
  version had. See Section 8 and the comparison demonstration
  immediately below it.
